In [7]:
# ==========================================================
# BLOQUE 1. IMPORTS Y MONTAJE DE DRIVE
#
# Solo se ejecuta una vez.
# ==========================================================

import os
import re
import json
import time
import unicodedata
import requests

from bs4 import BeautifulSoup

from urllib.parse import (
    urljoin,
    urlparse,
    urlunparse
)

from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:

# ==========================================================
# BLOQUE 2. RUTA DEL JSON EN EL DIRECTORIO EN EL QUE ESTAMOS
# ==========================================================

NOMBRE_PROGRAMA = "Extrae_Servicios.ipynb"  # ajusta al nombre real de tu notebook


ruta_programa = None

for root, dirs, files in os.walk("/content/drive/MyDrive"):

    if NOMBRE_PROGRAMA in files:

        ruta_programa = root

        break


if ruta_programa is None:

    raise Exception(
        "No se ha encontrado el notebook."
    )


CARPETA_JSON = os.path.join(

    ruta_programa,

    "JSONs"

)

os.makedirs(

    CARPETA_JSON,

    exist_ok=True

)

In [9]:
# ==========================================================
# BLOQUE 3. CONFIGURACIÓN DE LA FUENTE
#
# Fuente: Servicios universitarios - UPV
#
# A diferencia de "La institución", esta página no se
# organiza en varias secciones temáticas: es un único
# listado (buscador) de entidades. Por eso aquí solo hay
# una sección, "Servicios universitarios".
# ==========================================================

URL_SERVICIOS = (
    "https://www.upv.es/organizacion/"
    "servicios-universitarios/index-es.html"
)

NOMBRE_JSON = "servicios_universitarios.json"

RUTA_JSON = os.path.join(
    CARPETA_JSON,
    NOMBRE_JSON
)


# ----------------------------------------------------------
# Cabeceras HTTP (idénticas a las de institución)
# ----------------------------------------------------------

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/139.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,"
        "application/xml;q=0.9,image/avif,image/webp,"
        "*/*;q=0.8"
    ),
    "Accept-Language": (
        "es-ES,es;q=0.9,en;q=0.8"
    ),
    "Connection": "keep-alive"
}


# ==========================================================
# FUNCIONES AUXILIARES (idénticas a las de institución)
# ==========================================================

def limpiar_texto(texto):

    if texto is None:
        return ""

    texto = str(texto)

    texto = re.sub(
        r"\s+",
        " ",
        texto
    )

    return texto.strip()


def normalizar_identificador(texto):

    texto = limpiar_texto(texto)

    texto = unicodedata.normalize(
        "NFKD",
        texto
    )

    texto = "".join(
        caracter
        for caracter in texto
        if not unicodedata.combining(caracter)
    )

    texto = texto.lower()

    texto = re.sub(
        r"[^a-z0-9]+",
        "_",
        texto
    )

    return texto.strip("_")


def normalizar_url(
    url,
    url_base=URL_SERVICIOS
):

    if not url:
        return ""

    url = url.strip()

    return urljoin(
        url_base,
        url
    )


def es_url_valida(url):

    if not url:
        return False

    try:

        analisis = urlparse(url)

        return analisis.scheme in {
            "http",
            "https"
        }

    except Exception:

        return False


def deduplicar_lista(
    elementos,
    clave="url"
):

    resultado = []
    vistos = set()

    for elemento in elementos:

        if not isinstance(
            elemento,
            dict
        ):
            continue

        valor = elemento.get(
            clave,
            ""
        )

        if valor in vistos:
            continue

        vistos.add(valor)

        resultado.append(
            elemento
        )

    return resultado


def crear_seccion(
    titulo,
    tipo="seccion"
):

    return {

        "id": normalizar_identificador(
            titulo
        ),

        "titulo": limpiar_texto(
            titulo
        ),

        "tipo": tipo,

        "descripcion": "",

        "elementos": []
    }


def crear_elemento(
    titulo="",
    descripcion="",
    url="",
    tipo="recurso"
):

    return {

        "tipo": tipo,

        "titulo": limpiar_texto(
            titulo
        ),

        "descripcion": limpiar_texto(
            descripcion
        ),

        "url": normalizar_url(
            url
        )
    }


# ==========================================================
# ESTRUCTURA BASE DEL JSON
# ==========================================================

def crear_json_base():

    return {

        "titulo": "Servicios universitarios",

        "url": URL_SERVICIOS,

        "tipo": "padre",

        "secciones": [

            crear_seccion(
                "Servicios universitarios",
                "servicios_universitarios"
            )
        ]
    }


json_servicios = crear_json_base()


# ==========================================================
# RUTAS DE MARKDOWN
# ==========================================================

CARPETA_SERVICIOS = os.path.join(
    ruta_programa,
    "SERVICIOS_UNIVERSITARIOS"
)

CARPETA_SERVICIOS_RECURSOS = os.path.join(
    CARPETA_SERVICIOS,
    "servicios"
)

os.makedirs(
    CARPETA_SERVICIOS_RECURSOS,
    exist_ok=True
)

RUTA_MARKDOWN_PADRE_SERVICIOS = os.path.join(
    CARPETA_SERVICIOS,
    "servicios_universitarios.md"
)


print()
print("=" * 70)
print("ESTRUCTURA DE MARKDOWN - SERVICIOS UNIVERSITARIOS")
print("=" * 70)

print()
print("Carpeta principal:")
print(CARPETA_SERVICIOS)

print()
print("Markdown página padre:")
print(RUTA_MARKDOWN_PADRE_SERVICIOS)

print()
print("Carpeta de recursos:")
print(CARPETA_SERVICIOS_RECURSOS)

print()
print("OK: estructura de directorios preparada.")

print()
print("=" * 70)


ESTRUCTURA DE MARKDOWN - SERVICIOS UNIVERSITARIOS

Carpeta principal:
/content/drive/MyDrive/TFG Teleco/SERVICIOS_UNIVERSITARIOS

Markdown página padre:
/content/drive/MyDrive/TFG Teleco/SERVICIOS_UNIVERSITARIOS/servicios_universitarios.md

Carpeta de recursos:
/content/drive/MyDrive/TFG Teleco/SERVICIOS_UNIVERSITARIOS/servicios

OK: estructura de directorios preparada.



In [12]:
# ==========================================================
# BLOQUE 4. EXTRACCIÓN DE LOS SERVICIOS
#
# A diferencia de institución, aquí no hay que localizar
# varias tarjetas por encabezado: basta con recoger todos
# los enlaces a fichas de entidad (/entidades/...) dentro
# del contenedor principal.
#
# AVISO IMPORTANTE:
# Esta página carga resultados adicionales mediante
# JavaScript ("Cargar más resultados"). Con requests puro
# solo se obtienen los servicios ya presentes en el HTML
# inicial (normalmente unos 20 de los 74). Se avisa al
# final si el recuento es menor de lo esperado.
# ==========================================================

TOTAL_ESPERADO = 74


# ----------------------------------------------------------
# 1. Descargar la página
# ----------------------------------------------------------

respuesta = requests.get(
    URL_SERVICIOS,
    headers=HEADERS,
    timeout=30
)

respuesta.raise_for_status()

soup = BeautifulSoup(
    respuesta.text,
    "html.parser"
)


# ----------------------------------------------------------
# 2. Localizar el contenedor principal
# ----------------------------------------------------------

contenedor_principal = (
    soup.find(id="smooth-wrapper")
    or soup.find("main")
    or soup.body
)

if contenedor_principal is None:

    raise Exception(
        "No se ha encontrado el contenedor principal de la página."
    )


# ----------------------------------------------------------
# 3. Extraer los enlaces a entidades
# ----------------------------------------------------------

recursos = []

for enlace in contenedor_principal.find_all("a", href=True):

    href = enlace.get("href", "")

    if "/entidades/" not in href:
        continue

    titulo = limpiar_texto(
        enlace.get_text(
            " ",
            strip=True
        )
    )

    descripcion = limpiar_texto(
        enlace.get("title", "")
    )

    url = normalizar_url(
        href,
        URL_SERVICIOS
    )

    if not titulo or not es_url_valida(url):
        continue

    recursos.append(
        crear_elemento(
            titulo=titulo,
            descripcion=descripcion,
            url=url,
            tipo="recurso"
        )
    )

recursos = deduplicar_lista(
    recursos,
    clave="url"
)


# ----------------------------------------------------------
# 4. Construir el JSON
# ----------------------------------------------------------

seccion = crear_seccion(
    "Servicios universitarios",
    "servicios_universitarios"
)

seccion["elementos"] = recursos

json_servicios = {

    "titulo": "Servicios universitarios",

    "url": URL_SERVICIOS,

    "tipo": "padre",

    "secciones": [seccion]
}


# ----------------------------------------------------------
# 5. Mostrar resultado
# ----------------------------------------------------------

print()
print("=" * 70)
print("ESTRUCTURA EXTRAÍDA - SERVICIOS UNIVERSITARIOS")
print("=" * 70)

print()

for elemento in seccion["elementos"]:

    print(f"  - {elemento['titulo']}")
    print(f"    {elemento['url']}")

print()
print("Total de recursos encontrados:", len(seccion["elementos"]))
print("Total esperado:", TOTAL_ESPERADO)

if len(seccion["elementos"]) < TOTAL_ESPERADO:

    print()
    print(
        "AVISO: esta página carga el resto de resultados "
        "mediante JavaScript ('Cargar más resultados'). "
        "requests/BeautifulSoup no ejecutan ese JavaScript, "
        "así que solo se han recogido los servicios presentes "
        "en el HTML inicial. Consulta con el asistente la vía "
        "para completar el resto (endpoint AJAX vía DevTools)."
    )

print()
print("=" * 70)


ESTRUCTURA EXTRAÍDA - SERVICIOS UNIVERSITARIOS

  - Acción Cultural - (ACU)
    https://www.upv.es/entidades/ACU/index-es.html
  - Administración Electrónica y Transparencia - (SAET)
    https://www.upv.es/entidades/SAET/index-es.html
  - Agromuseu de Vera - (AGROMUSEU)
    https://www.upv.es/entidades/AGROMUSEU/index-es.html
  - Alumni - (ALUMNI)
    https://www.upv.es/entidades/ALUMNI/index-es.html
  - Área de Excelencia Académica - (AEXA)
    https://www.upv.es/entidades/AEXA/index-es.html
  - Asesoramiento a la I+D+i - (SAIDI)
    https://www.upv.es/entidades/SAIDI/index-es.html
  - Biblioteca y Documentación Científica - (ABDC)
    https://www.upv.es/entidades/ABDC/index-es.html
  - Calidad y Acreditación - (ACA)
    https://www.upv.es/entidades/ACA/index-es.html
  - Captación - Mecenazgo - (CAPT)
    https://www.upv.es/entidades/CAPT/index-es.html
  - Casa del Estudiante - (CALUM)
    https://www.upv.es/entidades/CALUM/index-es.html
  - Cátedras de Empresa - (CATEMPRE)
    https

In [13]:
# ==========================================================
# BLOQUE 5. GUARDAR EL JSON
# ==========================================================

with open(
    RUTA_JSON,
    "w",
    encoding="utf-8"
) as archivo:

    json.dump(
        json_servicios,
        archivo,
        ensure_ascii=False,
        indent=2
    )


print()
print("=" * 70)
print("JSON GUARDADO CORRECTAMENTE")
print("=" * 70)

print()
print("Archivo JSON:")
print(RUTA_JSON)

print()
print(
    "Número total de recursos:",
    sum(
        len(seccion["elementos"])
        for seccion in json_servicios["secciones"]
    )
)

print()
print("=" * 70)


JSON GUARDADO CORRECTAMENTE

Archivo JSON:
/content/drive/MyDrive/TFG Teleco/JSONs/servicios_universitarios.json

Número total de recursos: 78



In [11]:
# ==========================================================
# BLOQUE. EXTRACCIÓN DE MARKDOWN — SERVICIOS UNIVERSITARIOS
#
# Genera, en una carpeta nueva "SERVICIOS" dentro de la ruta
# del programa:
#
#   - servicios_universitarios.md   (página padre)
#   - un .md por cada servicio del JSON (visita su URL real
#     y extrae el contenido)
#
# Reutiliza el mismo motor de limpieza ya validado con "La
# institución": recorte desde el <h1> real, filtrado de
# menú/pie/breadcrumbs/RRSS/SQF genéricos de upv.es, corte en
# widgets de plantilla ("Esto te interesa", "Recursos",
# "Instalaciones", "Media") y deduplicado global.
#
# Antes de generar los .md, se filtran del JSON las entradas
# que NO son fichas reales de servicio (proceden del widget
# "Esto te interesa" o del pie de página y son duplicados de
# entradas ya presentes con el formato correcto):
#   - "Doctorados 30 programas en 8 ámbitos"
#   - "Deportes" (duplicado de "Deportes - (AD)")
#   - "Área de Comunicación" (duplicado de "Comunicación - (ACOM)")
#   - "Perfil del contratante" (duplicado de "Contratación - (CYO)")
# Se reconocen porque las fichas reales del listado siguen
# siempre el patrón "Nombre - (SIGLAS)"; lo que no sigue ese
# patrón no es un servicio propiamente dicho.
# ==========================================================

# ==========================================================
# BLOQUE 6. GENERACIÓN DE MARKDOWN (REESCRITO)
#
# Cambios respecto a la versión anterior:
#
# 1. Se corrige el cruce sección->carpeta usando "id" del
#    JSON en vez de "tipo" (ahí estaba el bug de
#    'organos_gobierno' que se saltaba entero).
# 2. La extracción de contenido ya no se limita a
#    h1/h2/h3/h4/p/li: recorre cualquier elemento "hoja"
#    (sin bloques anidados dentro), incluidos <a> sueltos,
#    para no perder nombres, teléfonos, emails y enlaces
#    "Más información". Los enlaces se convierten en
#    Markdown [texto](url) en vez de perder el href.
# 3. Se añaden metadatos YAML homogéneos en todos los .md,
#    con el formato exacto solicitado.
# 4. Si una página tiene poco contenido, se visitan hasta
#    MAX_ENLACES_HIJOS enlaces internos encontrados en el
#    contenedor principal y se añade un resumen de cada uno.
# 5. Si la URL no devuelve HTML (PDF, vídeo...), se genera
#    igualmente el .md con metadatos y un aviso, sin error.
# ==========================================================

import os
import re
import time
import unicodedata
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse


# ==========================================================
# 0. CONFIGURACIÓN DE METADATOS Y UMBRALES
# ==========================================================

FUENTE = "UPV"
CATEGORIA = "institucion"
NIVEL = "institucional"
PADRE_SLUG = "la_institucion"

UMBRAL_PALABRAS_POCO_CONTENIDO = 60
MAX_ENLACES_HIJOS = 5
MAX_CARACTERES_FRAGMENTO_HIJO = 800

TAGS_TITULO = {"h1", "h2", "h3", "h4", "h5", "h6"}

# Tags que consideramos "contenedores de bloque": si un
# elemento tiene alguno de estos como descendiente, NO es
# una hoja de contenido (para evitar duplicar texto).
TAGS_BLOQUE = {
    "h1", "h2", "h3", "h4", "h5", "h6",
    "p", "li", "div", "ul", "ol",
    "table", "section", "article", "blockquote"
}

# Tags que evaluamos como posibles hojas de contenido.
TAGS_CANDIDATAS = [
    "h1", "h2", "h3", "h4", "h5", "h6",
    "p", "li", "div", "span", "a"
]


# ==========================================================
# 1. FUNCIONES AUXILIARES YA EXISTENTES (sin cambios)
# ==========================================================

def nombre_archivo_markdown(titulo):
    return normalizar_identificador(titulo) + ".md"


def extraer_texto_limpio(elemento):
    if elemento is None:
        return ""
    texto = elemento.get_text(" ", strip=True)
    return limpiar_texto(texto)


def descargar_soup(url):
    """
    Descarga una página. Devuelve (soup, es_html).
    Si el recurso no es HTML (PDF, vídeo, etc.) es_html
    será False y soup será None.
    """

    respuesta = requests.get(
        url,
        headers=HEADERS,
        timeout=30
    )

    respuesta.raise_for_status()

    content_type = respuesta.headers.get("Content-Type", "")

    if "html" not in content_type.lower():
        return None, False

    return (
        BeautifulSoup(respuesta.text, "html.parser"),
        True
    )


def limpiar_contenido_html(soup):
    for elemento in soup.find_all(
        ["script", "style", "noscript", "svg", "nav", "footer", "header"]
    ):
        elemento.decompose()

    return soup


# ==========================================================
# 2. NUEVA EXTRACCIÓN DE CONTENIDO (hojas + enlaces)
# ==========================================================

def es_hoja_de_contenido(tag):
    """
    Un elemento es "hoja de contenido" si es candidato a
    contener texto propio y no tiene, a su vez, otros
    bloques anidados dentro (evitamos así procesar el mismo
    texto dos veces, una en el contenedor y otra en el hijo).
    """

    if tag.name not in TAGS_CANDIDATAS:
        return False

    if tag.name == "a":
        # Un <a> siempre se evalúa individualmente.
        return True

    descendientes_bloque = tag.find_all(TAGS_BLOQUE, recursive=True)

    return len(descendientes_bloque) == 0


def texto_markdown_de_elemento(tag, url_pagina):
    """
    Convierte el contenido de un elemento "hoja" a texto,
    respetando los enlaces que contenga como
    [texto](url) en vez de perder el href.
    """

    partes = []

    for nodo in tag.children:

        nombre_nodo = getattr(nodo, "name", None)

        if nombre_nodo == "a":

            texto_enlace = extraer_texto_limpio(nodo)
            href = (nodo.get("href") or "").strip()

            if not texto_enlace:
                continue

            if href and not href.startswith(("javascript:", "#")):

                href_absoluta = urljoin(url_pagina, href)
                href_absoluta = resolver_url_menu_antiguo(href_absoluta)

                partes.append(f"[{texto_enlace}]({href_absoluta})")

            else:

                partes.append(texto_enlace)

        elif nombre_nodo is not None:

            texto = extraer_texto_limpio(nodo)

            if texto:
                partes.append(texto)

        else:

            texto = limpiar_texto(str(nodo))

            if texto:
                partes.append(texto)

    return limpiar_texto(" ".join(partes))


def extraer_bloques_contenido(contenedor, url_pagina):
    """
    Recorre el contenedor y devuelve una lista de líneas
    Markdown, capturando títulos, párrafos, ítems de lista
    y enlaces sueltos (nombres, teléfonos, emails, "Más
    información"...).
    """

    lineas = []
    linea_anterior = None

    for tag in contenedor.find_all(TAGS_CANDIDATAS, recursive=True):

        if not es_hoja_de_contenido(tag):
            continue

        if tag.name == "a":

            texto = extraer_texto_limpio(tag)
            href = (tag.get("href") or "").strip()

            if not texto:
                continue

            if href and not href.startswith(("javascript:", "#")):

                href_absoluta = urljoin(url_pagina, href)
                href_absoluta = resolver_url_menu_antiguo(href_absoluta)
                linea = f"[{texto}]({href_absoluta})"

            else:

                linea = texto

        else:

            texto = texto_markdown_de_elemento(tag, url_pagina)

            if not texto:
                continue

            if tag.name in TAGS_TITULO:

                nivel = int(tag.name[1])
                linea = ("#" * nivel) + " " + texto

            elif tag.name == "li":

                linea = f"- {texto}"

            else:

                linea = texto

        # Evitamos duplicar la misma línea si aparece
        # justo después (artefacto habitual de anidamiento).
        if linea == linea_anterior:
            continue

        lineas.append(linea)
        linea_anterior = linea

    return lineas


def contar_palabras(lineas):
    return sum(len(linea.split()) for linea in lineas)


# ==========================================================
# 2bis. FILTRADO DE PLANTILLA (menú, migas, pie, widgets)
#
# Es el mismo menú y el mismo pie en TODO upv.es (plantilla
# nueva y antigua), así que esta lista sirve para cualquier
# sección que scrapees más adelante, no solo institución.
# ==========================================================

def normalizar_para_comparar(texto):

    texto = texto.strip().lower()
    texto = texto.strip("¡¿!?: ")

    texto = "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )

    texto = re.sub(r"\s+", " ", texto).strip()

    return texto


# Textos de menú, accesos rápidos y pie de página que
# aparecen igual en (casi) cualquier página de upv.es.
TEXTOS_BOILERPLATE = {
    "accesibilidad", "mapa web", "buscar", "directorio",
    "iniciar sesion", "emergencias", "inicio upv",
    "admision", "estudios", "investigacion", "organizacion",
    "comunidad upv",
    "admision a grado", "admision a master", "admision a doctorado",
    "internacional",
    "estudios de grado", "estudios de posgrado", "oferta academica",
    "estructuras de investigacion", "iniciativas de i+d+i", "innovacion",
    "la institucion", "vida universitaria", "escuelas y facultades",
    "departamentos", "servicios universitarios",
    "estudiante", "pas, pdi y pi", "ptgas, pdi y pi", "prensa", "titulados",
    "alumni upv", "orientador",
    "como llegar", "planos", "planos 2d", "contacto",
    "habla con nosotros",
    "tienes dudas", "contacta con nosotros",
    "quieres enviar una sugerencia, queja o felicitacion",
    "no has encontrado lo que buscas",
    "dinos que opinas", "consultanos",
    "sala de prensa", "noticias de la upv", "buscar un cargo docente",
    "area de comunicacion", "transparencia", "perfil del contratante",
    "aviso legal", "politica de cookies", "politica de privacidad",
    "gestion de cookies", "descarga nuestras apps",
    # Plantilla ANTIGUA de fichas de entidad (menú lateral de
    # idioma / accesibilidad / navegación, sin contenido real):
    "idioma", "idioma · language", "language",
    "valencia", "valencian", "english", "castellano",
    "cercar", "search", "directory", "directori",
    "contacte", "contact",
    "otros", "donde estamos", "¿donde estamos?",
    "webs relacionadas",
}

# Estos solo cuentan como corte de plantilla si aparecen
# como TÍTULO (línea que empieza por #), para no arriesgarnos
# a cortar contenido real que use esa misma palabra suelta.
TITULOS_CORTE_PLANTILLA = {
    "esto te interesa", "recursos", "instalaciones", "media",
}


def es_linea_boilerplate(linea):

    texto_plano = re.sub(r"^#+\s*", "", linea)
    texto_plano = re.sub(r"^-\s*", "", texto_plano)

    match_enlace = re.match(r"^\[([^\]]+)\]\([^)]+\)$", texto_plano.strip())
    if match_enlace:
        texto_plano = match_enlace.group(1)

    normalizado = normalizar_para_comparar(texto_plano)

    if normalizado in TEXTOS_BOILERPLATE:
        return True

    if "universitat politecnica de valencia" in normalizado and "©" in linea:
        return True

    if re.match(r"^tel\.?\s*\(?\+?34", normalizado):
        return True

    if "::" in linea:
        return True

    if linea.startswith("#") and normalizado in TITULOS_CORTE_PLANTILLA:
        # Señal de corte: todo lo que va desde aquí es pie de
        # página / widget promocional. Se gestiona en
        # recortar_desde_titulo_plantilla, aquí solo la marcamos.
        return True

    return False


def recortar_desde_primer_h1(lineas):
    """
    El contenido real de cualquier página de upv.es (plantilla
    nueva o antigua) empieza en su <h1>. Todo lo anterior es
    menú, migas de pan o barra lateral de navegación.
    """

    for indice, linea in enumerate(lineas):

        if linea.startswith("# "):
            return lineas[indice:]

    return lineas


def recortar_en_titulo_plantilla(lineas):
    """
    Corta el documento en cuanto aparece un título que marca
    el inicio de pie de página / widgets promocionales
    ("¡Esto te interesa!", "Recursos", "Instalaciones", "Media").
    """

    for indice, linea in enumerate(lineas):

        if not linea.startswith("#"):
            continue

        texto_titulo = re.sub(r"^#+\s*", "", linea)
        normalizado = normalizar_para_comparar(texto_titulo)

        if normalizado in TITULOS_CORTE_PLANTILLA:
            return lineas[:indice]

    return lineas


def deduplicar_global(lineas):
    """
    Elimina líneas exactamente repetidas en todo el documento
    (no solo consecutivas), útil para plantillas antiguas que
    duplican el pie de página completo.
    """

    resultado = []
    vistos = set()

    for linea in lineas:

        if linea in vistos:
            continue

        vistos.add(linea)
        resultado.append(linea)

    return resultado


def limpiar_lineas_finales(lineas, recortar_h1=True):
    """
    Aplica, en orden: recorte a partir del primer <h1> (salvo
    que recortar_h1=False, útil en la página padre que ya
    gestiona su propio título), corte en los títulos de pie
    de página / widgets promocionales, filtrado de líneas de
    menú/pie sueltas, y deduplicado global. "Sabías que" NO
    se filtra (contiene datos reales de la UPV); solo se
    corta todo lo que va después de "¡Esto te interesa!",
    "Recursos", "Instalaciones" o "Media".
    """

    if recortar_h1:
        lineas = recortar_desde_primer_h1(lineas)

    lineas = recortar_en_titulo_plantilla(lineas)

    lineas = [
        linea for linea in lineas
        if not es_linea_boilerplate(linea)
    ]

    lineas = deduplicar_global(lineas)

    return lineas


# ==========================================================
# 3. RUTAS DE SALIDA (carpeta plana "SERVICIOS")
# ==========================================================

CARPETA_SERVICIOS = os.path.join(ruta_programa, "SERVICIOS")
os.makedirs(CARPETA_SERVICIOS, exist_ok=True)

RUTA_MARKDOWN_PADRE_SERVICIOS = os.path.join(
    CARPETA_SERVICIOS, "servicios_universitarios.md"
)

# Reutiliza RUTA_JSON si ya está definida de un bloque
# anterior en esta misma sesión; si no, apunta al fichero
# guardado en su momento.
if "RUTA_JSON" not in dir():
    RUTA_JSON = os.path.join(CARPETA_JSON, "servicios_universitarios.json")

URL_SERVICIOS = "https://www.upv.es/organizacion/servicios-universitarios/index-es.html"


# ==========================================================
# 4. CARGAR Y LIMPIAR EL JSON
# ==========================================================

with open(RUTA_JSON, "r", encoding="utf-8") as archivo:
    json_servicios = json.load(archivo)

PATRON_FICHA_SERVICIO = re.compile(r".+\s-\s\([A-Za-z0-9]+\)\s*$")


def es_ficha_real_de_servicio(elemento):
    return bool(PATRON_FICHA_SERVICIO.match(elemento.get("titulo", "").strip()))


def codigo_entidad(url):
    match = re.search(r"/entidades/([A-Za-z0-9_\-]+)/?", url)
    return match.group(1).upper() if match else url


seccion_servicios = json_servicios["secciones"][0]
elementos_originales = seccion_servicios["elementos"]

elementos_validos = [e for e in elementos_originales if es_ficha_real_de_servicio(e)]
elementos_descartados = [e for e in elementos_originales if not es_ficha_real_de_servicio(e)]

# Deduplicado adicional por código de entidad (por si hubiera
# dos fichas reales apuntando a la misma entidad).
elementos_limpios = []
codigos_vistos = set()

for elemento in elementos_validos:
    codigo = codigo_entidad(elemento["url"])
    if codigo in codigos_vistos:
        continue
    codigos_vistos.add(codigo)
    elementos_limpios.append(elemento)

seccion_servicios["elementos"] = elementos_limpios

print()
print("=" * 70)
print("LIMPIEZA DEL JSON DE SERVICIOS")
print("=" * 70)
print()
print("Elementos originales en el JSON:", len(elementos_originales))
print("Descartados (no son ficha real de servicio):", len(elementos_descartados))

for e in elementos_descartados:
    print(f"  - {e['titulo']}  ->  {e['url']}")

print()
print("Servicios finales a procesar:", len(elementos_limpios))
print("=" * 70)


# ==========================================================
# 5. EXPANSIÓN DE ENLACES HIJOS (páginas con poco contenido)
# ==========================================================

def es_url_valida_para_expandir(href, url_pagina, urls_ya_usadas):

    if not href:
        return False

    href = href.strip()

    if href.startswith(("mailto:", "tel:", "javascript:", "#")):
        return False

    absoluta = urljoin(url_pagina, href).split("#")[0]

    if absoluta == url_pagina.split("#")[0]:
        return False

    if absoluta in urls_ya_usadas:
        return False

    dominio = urlparse(absoluta).netloc

    if "upv.es" not in dominio:
        return False

    return True


# Patrones de URL que nunca aportan contenido propio de la
# entidad: selectores de idioma, accesibilidad, buscador
# genérico, directorio de personas, plano, "cómo llegar"...
# Son utilidades de plantilla de TODO upv.es, no información
# específica de este servicio.
PATRONES_URL_EXCLUIDOS_HIJOS = [
    r"/bin2/tipoacc/",                    # accesibilidad (a / A)
    r"sic_mag\.MetaBus",                  # buscador genérico UPV
    r"/plano/plano-2d",                   # planos del campus
    r"/otros/como-llegar",                # cómo llegar
    r"index-va\.html?$",                  # versión en valenciano
    r"index-en\.html?$",                  # versión en inglés
    r"index-i\.html?$",
    r"index-v\.html?$",
    r"/otros/accesibilidad",
    r"/otros/mapa-web",
    r"/otros/contacto",
]


def es_directorio_generico_de_personas(href):
    """
    'sic_per.Busca_Persona' se usa tanto para el Directorio
    genérico de toda la UPV (sin parámetros de entidad) como
    para el "Equipo directivo" de una entidad concreta (con
    P_SG=.../P_CARGOS=...). Solo el primero es ruido; el
    segundo es contenido propio de la ficha y no se descarta.
    """

    if "sic_per.Busca_Persona" not in href:
        return False

    return "P_SG=" not in href and "P_CARGOS=" not in href


def es_variante_de_la_misma_pagina(href_absoluta, url_pagina):
    """
    En la plantilla antigua, el propio menú lateral incluye un
    enlace con el nombre de la entidad que apunta a un simple
    alias de la misma página (p.ej. 'indexc.html' junto a
    'index-es.html'), sin aportar nada nuevo. Se detecta
    comparando el código de entidad de ambas URLs: si es el
    mismo Y el enlace es solo otra variante de "index", se
    descarta como autorreferencia.
    """

    if codigo_entidad(href_absoluta) != codigo_entidad(url_pagina):
        return False

    return bool(re.search(r"/index\w*\.html?$", href_absoluta, flags=re.IGNORECASE))


def resolver_url_menu_antiguo(url_absoluta):
    """
    La plantilla antigua de fichas de entidad usa enlaces de
    menú tipo:

        .../menu_urlc.html?//www.upv.es/pls/oalu/sic_infoent.InfoGeneralMS?P_ENTIDAD=CORR...

    'menu_urlX.html' es solo una cáscara: la URL real del
    contenido va incrustada en la query string, tras '?//'.
    Un navegador la lee con JavaScript y la carga; requests no
    ejecuta ese JavaScript, así que sin esto siempre se
    descarga la misma página vacía del menú, se siga el
    enlace que se siga. Aquí se extrae y se sigue la URL real.
    """

    coincidencia = re.search(r"menu_url\w*\.html\?(//.+)$", url_absoluta, flags=re.IGNORECASE)

    if not coincidencia:
        return url_absoluta

    interno = coincidencia.group(1)

    if interno.startswith("//"):
        interno = "https:" + interno

    return interno


def extraer_texto_y_url_de_linea_markdown(linea):

    coincidencia = re.match(r"^\[([^\]]*)\]\(([^)]+)\)$", linea.strip())

    if not coincidencia:
        return None, None

    return coincidencia.group(1), coincidencia.group(2)


def es_autorreferencia_o_accesibilidad(linea, url_pagina):
    """
    Filtro adicional para el contenido PRINCIPAL (no solo para
    la selección de enlaces hijos): descarta líneas que son
    únicamente un enlace de accesibilidad (texto de 1-2
    caracteres, p.ej. "a" / "A") o una autorreferencia a la
    misma página con otro nombre (p.ej. 'indexc.html' junto a
    'index-es.html').
    """

    texto, href = extraer_texto_y_url_de_linea_markdown(linea)

    if href is None:
        return False

    if len(texto.strip()) <= 2:
        return True

    return es_variante_de_la_misma_pagina(href, url_pagina)


# Textos de enlace que tampoco aportan nada propio de la
# entidad (idiomas, "buscar", "directorio"...), aunque la URL
# no encaje con los patrones anteriores.
TEXTOS_EXCLUIDOS_HIJOS = {
    "valencia", "valencia language", "valencian", "english",
    "castellano", "cercar", "search", "directory", "directori",
    "contacte", "contact", "idioma", "language", "idioma language",
}

# En la plantilla antigua, estos son los enlaces del menú
# lateral que SÍ llevan a contenido real de la entidad. Se
# priorizan para no desperdiciar el cupo de enlaces hijos en
# autorreferencias u otro ruido de plantilla.
TEXTOS_PRIORITARIOS_HIJOS = [
    "informacion general", "quienes somos", "presentacion",
    "equipo directivo", "webs relacionadas", "servicios",
    "tramites", "memoria", "funciones", "organigrama",
]


def es_enlace_hijo_util(texto, href, url_pagina):

    texto_normalizado = normalizar_para_comparar(texto)

    # Textos de 1-2 caracteres son casi siempre botones de
    # accesibilidad (tamaño de letra "a" / "A"), no contenido.
    if len(texto.strip()) <= 2:
        return False

    if texto_normalizado in TEXTOS_BOILERPLATE:
        return False

    if texto_normalizado in TEXTOS_EXCLUIDOS_HIJOS:
        return False

    if es_directorio_generico_de_personas(href):
        return False

    absoluta = urljoin(url_pagina, href).split("#")[0]
    absoluta = resolver_url_menu_antiguo(absoluta)

    if es_variante_de_la_misma_pagina(absoluta, url_pagina):
        return False

    for patron in PATRONES_URL_EXCLUIDOS_HIJOS:
        if re.search(patron, href, flags=re.IGNORECASE):
            return False

    return True


def obtener_enlaces_hijos(contenedor, url_pagina, maximo=MAX_ENLACES_HIJOS):

    candidatos = []
    urls_vistas = set()

    for a in contenedor.find_all("a", href=True):

        href = a["href"]

        if not es_url_valida_para_expandir(href, url_pagina, urls_vistas):
            continue

        texto = extraer_texto_limpio(a)

        if not texto:
            continue

        if not es_enlace_hijo_util(texto, href, url_pagina):
            continue

        absoluta = urljoin(url_pagina, href).split("#")[0]
        absoluta = resolver_url_menu_antiguo(absoluta)

        urls_vistas.add(absoluta)
        candidatos.append((texto, absoluta))

    # Priorizamos los enlaces que en la plantilla antigua
    # llevan a contenido real (Información general, Equipo
    # directivo...), para no gastar el cupo en lo primero que
    # aparezca en el menú.
    def prioridad(candidato):
        texto_normalizado = normalizar_para_comparar(candidato[0])
        return 0 if texto_normalizado in TEXTOS_PRIORITARIOS_HIJOS else 1

    candidatos.sort(key=prioridad)

    return candidatos[:maximo]

    return enlaces


def resumir_pagina_hija(url):

    try:

        soup, es_html = descargar_soup(url)

        if not es_html:
            return None

        soup = limpiar_contenido_html(soup)

        contenedor = (
            soup.find(id="smooth-wrapper")
            or soup.find("main")
            or soup.body
        )

        if contenedor is None:
            return None

        lineas = extraer_bloques_contenido(contenedor, url)
        lineas = limpiar_lineas_finales(lineas)
        lineas = [l for l in lineas if not es_autorreferencia_o_accesibilidad(l, url)]

        fragmento = "\n\n".join(lineas)

        if len(fragmento) > MAX_CARACTERES_FRAGMENTO_HIJO:
            fragmento = fragmento[:MAX_CARACTERES_FRAGMENTO_HIJO].rstrip() + "…"

        return fragmento or None

    except Exception:

        return None


# ==========================================================
# 6. METADATOS YAML
# ==========================================================

FUENTE = "UPV"
CATEGORIA = "servicios_universitarios"
NIVEL = "institucional"
PADRE_SLUG = "servicios_universitarios"


def generar_yaml_metadatos(seccion_id, recurso, tipo_documento="recurso", tipo_recurso="servicio"):

    campos = [
        ("fuente", FUENTE),
        ("categoria", CATEGORIA),
        ("nivel", NIVEL),
        ("tipo_documento", tipo_documento),
        ("tipo_recurso", tipo_recurso),
        ("padre", PADRE_SLUG),
        ("seccion", seccion_id),
        ("url", recurso.get("url", "")),
    ]

    lineas = [f"{clave}: {valor}" for clave, valor in campos]

    return "---\n" + "\n\n".join(lineas) + "\n---\n"


# ==========================================================
# 7. GENERAR MARKDOWN DE CADA SERVICIO
# ==========================================================

def generar_markdown_recurso(recurso, carpeta, seccion_id):

    titulo = recurso.get("titulo", "")
    url = recurso.get("url", "")

    if not titulo or not url:
        return False

    print()
    print(f"  Extrayendo: {titulo}")
    print(f"  URL: {url}")

    try:

        soup, es_html = descargar_soup(url)

        yaml_metadatos = generar_yaml_metadatos(seccion_id, recurso)

        if not es_html:

            markdown = (
                f"{yaml_metadatos}\n"
                f"# {titulo}\n\n"
                f"**URL:** {url}\n\n"
                f"_Este recurso no es una página HTML estándar "
                f"(por ejemplo, un PDF o un vídeo). "
                f"Consulta el contenido directamente en la URL indicada._\n"
            )

            nombre_archivo = nombre_archivo_markdown(titulo)
            ruta_archivo = os.path.join(carpeta, nombre_archivo)

            with open(ruta_archivo, "w", encoding="utf-8") as archivo:
                archivo.write(markdown)

            print(f"  OK (no HTML): {ruta_archivo}")

            return True

        soup = limpiar_contenido_html(soup)

        contenido = (
            soup.find(id="smooth-wrapper")
            or soup.find("main")
            or soup.body
        )

        if contenido is None:
            print("  AVISO: no se ha encontrado contenido.")
            return False

        lineas_contenido = extraer_bloques_contenido(contenido, url)
        lineas_contenido = limpiar_lineas_finales(lineas_contenido)
        lineas_contenido = [
            l for l in lineas_contenido
            if not es_autorreferencia_o_accesibilidad(l, url)
        ]

        if not lineas_contenido:
            print("  AVISO: contenido vacío.")
            return False

        if contar_palabras(lineas_contenido) < UMBRAL_PALABRAS_POCO_CONTENIDO:

            enlaces_hijos = obtener_enlaces_hijos(contenido, url)

            if enlaces_hijos:

                lineas_contenido.append("## Información relacionada")

                for texto_enlace, url_hija in enlaces_hijos:

                    time.sleep(0.5)

                    fragmento = resumir_pagina_hija(url_hija)

                    lineas_contenido.append(f"### {texto_enlace}")
                    lineas_contenido.append(f"**URL:** {url_hija}")

                    if fragmento:
                        lineas_contenido.append(fragmento)

        markdown_contenido = "\n\n".join(lineas_contenido)

        descripcion = recurso.get("descripcion", "").strip()

        bloque_descripcion = (
            f"**Descripción breve:** {descripcion}\n\n"
            if descripcion else ""
        )

        markdown = (
            f"{yaml_metadatos}\n"
            f"# {titulo}\n\n"
            f"**URL:** {url}\n\n"
            f"{bloque_descripcion}"
            f"{markdown_contenido}\n"
        )

        nombre_archivo = nombre_archivo_markdown(titulo)
        ruta_archivo = os.path.join(carpeta, nombre_archivo)

        with open(ruta_archivo, "w", encoding="utf-8") as archivo:
            archivo.write(markdown)

        print(f"  OK: {ruta_archivo}")

        return True

    except Exception as error:

        print(f"  ERROR: {error}")
        return False


# ==========================================================
# 8. GENERAR MARKDOWN DE LA PÁGINA PADRE
#
# La página índice incrusta el listado de servicios en el
# mismo contenedor que el texto descriptivo, así que aquí se
# filtran explícitamente los enlaces a /entidades/... (ya son
# documentos propios) y el texto del widget de filtros de
# esta página en concreto.
# ==========================================================

FILTROS_LISTADO_SERVICIOS = {
    "volver a resultados", "filtrar por", "fundaciones upv",
    "servicios generales", "filtrar", "buscar por palabras",
    "necesitas escribir un minimo de 3 caracteres",
    "cargar mas resultados", "resultados",
}


def es_enlace_a_entidad(linea):
    return bool(re.search(r"\]\(https?://(www\.)?upv\.es/entidades/", linea))


def es_ruido_listado_servicios(linea):

    if es_enlace_a_entidad(linea):
        return True

    texto_plano = re.sub(r"^#+\s*", "", linea)
    texto_plano = re.sub(r"^-\s*", "", texto_plano)

    normalizado = normalizar_para_comparar(texto_plano)

    if normalizado in FILTROS_LISTADO_SERVICIOS:
        return True

    if re.match(r"^mostrando resultados de", normalizado):
        return True

    return False


def generar_markdown_padre_servicios():

    print()
    print("=" * 70)
    print("GENERANDO MARKDOWN DE LA PÁGINA PADRE (SERVICIOS)")
    print("=" * 70)

    soup, es_html = descargar_soup(URL_SERVICIOS)
    soup = limpiar_contenido_html(soup)

    contenedor = (
        soup.find(id="smooth-wrapper")
        or soup.find("main")
        or soup.body
    )

    if contenedor is None:
        raise Exception("No se ha encontrado el contenedor principal.")

    lineas = extraer_bloques_contenido(contenedor, URL_SERVICIOS)
    lineas = limpiar_lineas_finales(lineas)

    lineas = [linea for linea in lineas if not es_ruido_listado_servicios(linea)]
    lineas = deduplicar_global(lineas)

    yaml_metadatos = generar_yaml_metadatos(
        seccion_id="servicios_universitarios",
        recurso={"url": URL_SERVICIOS},
        tipo_documento="padre",
        tipo_recurso="informacion",
    )

    markdown = (
        f"{yaml_metadatos}\n"
        + "\n\n".join(lineas)
        + "\n"
    )

    with open(RUTA_MARKDOWN_PADRE_SERVICIOS, "w", encoding="utf-8") as archivo:
        archivo.write(markdown)

    print()
    print("OK: Markdown padre generado:")
    print(RUTA_MARKDOWN_PADRE_SERVICIOS)

    return markdown


# ==========================================================
# 9. EJECUCIÓN
# ==========================================================

markdown_padre_servicios = generar_markdown_padre_servicios()

total_servicios = 0
servicios_correctos = 0
servicios_error = 0

print()
print("=" * 70)
print("GENERANDO MARKDOWNS DE SERVICIOS UNIVERSITARIOS")
print("=" * 70)

for recurso in seccion_servicios["elementos"]:

    total_servicios += 1

    if generar_markdown_recurso(recurso, CARPETA_SERVICIOS, seccion_servicios["id"]):
        servicios_correctos += 1
    else:
        servicios_error += 1


# ==========================================================
# 10. RESUMEN FINAL
# ==========================================================

print()
print("=" * 70)
print("GENERACIÓN DE MARKDOWN FINALIZADA")
print("=" * 70)

print()
print("Markdown página padre:")
print(RUTA_MARKDOWN_PADRE_SERVICIOS)

print()
print("Servicios procesados:", total_servicios)
print("Markdowns generados:", servicios_correctos)
print("Errores:", servicios_error)

print()
print("Total de documentos en la carpeta SERVICIOS:", servicios_correctos + 1)

print()
print("Carpeta de salida:")
print(CARPETA_SERVICIOS)

print()
print("=" * 70)


LIMPIEZA DEL JSON DE SERVICIOS

Elementos originales en el JSON: 78
Descartados (no son ficha real de servicio): 4
  - Doctorados 30 programas en 8 ámbitos  ->  https://www.upv.es/entidades/EDOCTORADO/info/1007774normalc.html
  - Deportes  ->  http://www.upv.es/entidades/AD/
  - Área de Comunicación  ->  http://www.upv.es/entidades/ACOM/index-es.html
  - Perfil del contratante  ->  http://www.upv.es/entidades/CYO

Servicios finales a procesar: 74

GENERANDO MARKDOWN DE LA PÁGINA PADRE (SERVICIOS)

OK: Markdown padre generado:
/content/drive/MyDrive/TFG Teleco/SERVICIOS/servicios_universitarios.md

GENERANDO MARKDOWNS DE SERVICIOS UNIVERSITARIOS

  Extrayendo: Acción Cultural - (ACU)
  URL: https://www.upv.es/entidades/ACU/index-es.html
  OK: /content/drive/MyDrive/TFG Teleco/SERVICIOS/accion_cultural_acu.md

  Extrayendo: Administración Electrónica y Transparencia - (SAET)
  URL: https://www.upv.es/entidades/SAET/index-es.html
  OK: /content/drive/MyDrive/TFG Teleco/SERVICIOS/administ